# Deep Gaussian Processes for Crypto Portfolio Allocation

In [ ]:
import torch
import tqdm
import math
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
import sqlite3
from huggingface_hub import hf_hub_download
from gpytorch.settings import linalg_dtypes
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

import gpytorch
from torch.nn import Linear
from gpytorch.means import ConstantMean, LinearMean
from gpytorch.kernels import RBFKernel, ScaleKernel
from gpytorch.variational import VariationalStrategy, CholeskyVariationalDistribution, LMCVariationalStrategy
from gpytorch.distributions import MultivariateNormal
from gpytorch.models.deep_gps import DeepGPLayer, DeepGP
from gpytorch.mlls import DeepApproximateMLL, VariationalELBO
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.models.deep_gps import DeepGPLayer, DeepGP
from gpytorch.mlls import DeepApproximateMLL

# Data Compiling 

In [2]:
conn = sqlite3.connect('dydx_market_data.db')
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

table_name = tables[3][0]

print("Using table:", table_name)

cursor.execute(f"PRAGMA table_info({table_name});")

columns = []
for col in cursor.fetchall():
    columns.append(col[1])

cursor.execute(f"SELECT COUNT(*) FROM {table_name}")

cursor.execute(f"SELECT * FROM {table_name}")

#Select data 
rows = cursor.fetchall()
conn.close()

df = pd.DataFrame(rows)
df.columns = columns

Using table: historical_funding


In [3]:
count = df['market_id'].unique()

#Get asset names and store into dict indexed by asset name 
df_dict = dict()
for i in count:
    df_dict[i] = df[df['market_id'] == i]

# Deep Gaussian Process 

In [4]:
#Define hidden layer class
class HiddenLayerClass(DeepGPLayer):
    def __init__(self, input_size, output_size, inducing_points = 64, mean_type = "constant"):
        #Init number of inducing points between input and output size randomly
        if output_size is None:
            self.inducing_points = torch.randn(inducing_points, input_size)
            self.batch_shape = torch.Size([])
        
        else:    
            self.inducing_points =  torch.randn(output_size, inducing_points, input_size)
            self.batch_shape = torch.Size([output_size])

        variational_distribution = CholeskyVariationalDistribution(num_inducing_points= inducing_points,
            batch_shape=self.batch_shape)
        
        variational_strategy = VariationalStrategy(
            self,
            self.inducing_points,
            variational_distribution,
            learn_inducing_locations=True
        )
        super(HiddenLayerClass, self).__init__(variational_strategy, input_size, output_size)

        if mean_type == 'constant':
            self.mean_module = ConstantMean(batch_shape=self.batch_shape)
            self.mean_module.initialize(constant = 0.0)
        else:
            #Heteroskedastic regime 
            self.mean_module = LinearMean(input_size)
        self.covar_module = ScaleKernel(
            RBFKernel(batch_shape=self.batch_shape, ard_num_dims=input_size),
            batch_shape=self.batch_shape, ard_num_dims=None)
            
    #Forward pass with multivariate normal with mean and covar params 
    def forward(self, x):
            mean_x = self.mean_module(x)
            covar_x = self.covar_module(x)
            return MultivariateNormal(mean_x, covar_x)

## Define training data for each crypto-asset

In [5]:
train_dict, test_dict = dict(), dict()

for asset in df_dict:
    rate_price =  pd.concat([df_dict[asset]['rate'], df_dict[asset]['price']], axis = 1)
    #Ensure that shuffle is set to false, else causality is destroyed 
    X_train, X_test = train_test_split(rate_price, test_size=0.2, shuffle = False)
    train_dict[asset] = X_train
    test_dict[asset] = X_test

#X_train and X_test contain test-training splits for EACH crypto-asset that will be fed into multiple DGPs

In [6]:
hidden_dim = 10

class DeepGaussianProcess(DeepGP):
    def __init__(self, input_dim):
        
        hidden_layer = HiddenLayerClass(input_dim, hidden_dim, mean_type = "Linear")
        #Output of none is critical as it collapses to scalar
        last_layer = HiddenLayerClass(hidden_dim, None, mean_type='constant')
        
        super().__init__()
        self.hidden_layer = hidden_layer
        self.last_layer = last_layer
        self.likelihood = GaussianLikelihood()


    def forward(self, inputs):
        #Forward-pass identical to MLP
        hidden = self.hidden_layer(inputs)
        output = self.last_layer(hidden)
        return output

    def predict(data):
        with torch.no_grad():
            mus = []
            variances = []
            lls = []
            for x_batch, y_batch in data:
                preds = self.likelihood(self(x_batch))
                mus.append(preds.mean)
                variances.append(preds.variance)
                lls.append(model.likelihood.log_marginal(y_batch, model(x_batch)))
        
        return torch.cat(mus, dim=-1), torch.cat(variances, dim=-1), torch.cat(lls, dim=-1)

## Training loop

In [7]:
#Training setup
num_epochs = 50
num_samples = 16
device = torch.device("cpu")
torch.set_num_threads(10)
torch.set_num_interop_threads(4)

model_dict = dict()
for j in count:
    print(j)
    
    train_x =  train_dict[j]
    returns = np.diff(np.log(train_x['price'].astype('float').values))
    
    X = np.column_stack([train_x['rate'].astype('float').values[:-1], train_x['price'].astype('float').values[:-1]])
    
    #Organize data types for passing into DGP
    Y_tensor = torch.tensor(returns, dtype = torch.float32)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_tensor = torch.tensor(X_scaled, dtype = torch.float32)
    
    
    train_dataset = TensorDataset(X_tensor, Y_tensor)
    #Again make sure shuffle is set to false here
    train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=False)
 
    # Funding rate and price are inputs
    model = DeepGaussianProcess(2)
    model = model.to(device)
    likelihood = gpytorch.likelihoods.GaussianLikelihood()
    likelihood = likelihood.to(device)
    
    optimizer = torch.optim.Adam([
        {'params': model.parameters()}, {'params': likelihood.parameters()},
    ], lr=3e-3)
    
    mll = DeepApproximateMLL(VariationalELBO(model.likelihood, model, num_data = len(Y_tensor)))
    
    epochs_iter = tqdm.notebook.tqdm(range(num_epochs), desc="Epoch")
    for i in epochs_iter:
        epoch_loss = 0.0
        # Within each iteration, we will go over each minibatch of data
        minibatch_iter = tqdm.notebook.tqdm(train_loader, desc="Minibatch", leave=False)
        for x_batch, y_batch in minibatch_iter:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            with gpytorch.settings.num_likelihood_samples(num_samples):
                optimizer.zero_grad()
                output = model(x_batch)
                loss = -mll(output, y_batch)
                loss.backward()
                optimizer.step()
    
                minibatch_iter.set_postfix(loss=loss.item())
    model_dict[j] = model, scaler, likelihood

1INCH-USD


Epoch:   0%|          | 0/50 [00:00<?, ?it/s]

Minibatch:   0%|          | 0/10 [00:00<?, ?it/s]

Minibatch:   0%|          | 0/10 [00:00<?, ?it/s]

Minibatch:   0%|          | 0/10 [00:00<?, ?it/s]

...

Minibatch:   0%|          | 0/8 [00:00<?, ?it/s]

Minibatch:   0%|          | 0/8 [00:00<?, ?it/s]

Minibatch:   0%|          | 0/8 [00:00<?, ?it/s]

Minibatch:   0%|          | 0/8 [00:00<?, ?it/s]

Minibatch:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")

In [ ]:
# Walk-forward configuration
TRAIN_WINDOW = 15000
TEST_WINDOW = 2500
N_FOLDS = 4
BATCH_SIZE = 256
NUM_EPOCHS = 100
NUM_SAMPLES = 16
LR = 3e-3


device = torch.device("cpu")
torch.set_num_threads(8)

# Results storage
fold_portfolio_returns = []
fold_weights_list = []

for fold in range(N_FOLDS):
    print("\n" + "=" * 50)
    print(f"FOLD {fold + 1}/{N_FOLDS}")
    
    train_start = fold * TEST_WINDOW
    train_end = train_start + TRAIN_WINDOW
    test_start = train_end
    test_end = test_start + TEST_WINDOW
    
    print(f"Train: [{train_start} : {train_end}]")
    print(f"Test:  [{test_start} : {test_end}]")
    
    fold_means = []
    fold_vars = []
    fold_returns = []

    for j in count:
        if j == "LUNA-USD":
            continue
    
        if j not in model_dict:
            print(f"Skipping {j} - not in model_dict")
            continue
    
        try:
            full_data = train_dict[j]
            prices = full_data["price"].astype(float).values
            rates = full_data["rate"].astype(float).values
    
            returns = np.diff(np.log(prices))
            X = np.column_stack([rates[:-1], prices[:-1]])
    
            X_train_raw = X[train_start:train_end]
            y_train = returns[train_start:train_end]
            X_test_raw = X[test_start:test_end]
            y_test = returns[test_start:test_end]
    
            if len(X_train_raw) < 1000 or len(X_test_raw) < 100:
                print(f"Skipping {j} - insufficient data in fold")
                continue
    
            fold_scaler = StandardScaler()
            X_train_scaled = fold_scaler.fit_transform(X_train_raw).astype(np.float32)
            X_test_scaled = fold_scaler.transform(X_test_raw).astype(np.float32)
            y_train = y_train.astype(np.float32)
            y_test = y_test.astype(np.float32)
    
            X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
            y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
            X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
            y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

        except Exception as e:
            print(f"Skipping {j} - data error: {e}")
            continue
    
        if (
            torch.isnan(X_train_tensor).any()
            or torch.isnan(y_train_tensor).any()
            or torch.isnan(X_test_tensor).any()
            or torch.isnan(y_test_tensor).any()
        ):
            print(f"Skipping {j} - NaN in data")
            continue
    
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
        fold_model = DeepGaussianProcess(input_dim=2).float().to(device)
        fold_likelihood = gpytorch.likelihoods.GaussianLikelihood().float().to(device)
    
        optimizer = torch.optim.Adam(
            [
                {"params": fold_model.parameters()},
                {"params": fold_likelihood.parameters()},
            ],
            lr=LR,
        )
    
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=20,
            gamma=0.5,
        )
    
        mll = gpytorch.mlls.DeepApproximateMLL(
            gpytorch.mlls.VariationalELBO(
                fold_likelihood,
                fold_model,
                num_data=len(y_train_tensor),
            )
        )
    
        fold_model.train()
        fold_likelihood.train()
    
        for epoch in range(NUM_EPOCHS):
            epoch_loss = 0.0
    
            for x_batch, y_batch in train_loader:
                x_batch = x_batch.to(device=device, dtype=torch.float32)
                y_batch = y_batch.to(device=device, dtype=torch.float32)
    
                optimizer.zero_grad()
    
                with gpytorch.settings.num_likelihood_samples(NUM_SAMPLES):
                    output = fold_model(x_batch)
                    loss = -mll(output, y_batch)
    
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
    
            scheduler.step()
    
            if epoch % 10 == 0:
                avg_loss = epoch_loss / len(train_loader)
                print(f"  {j} epoch {epoch}: loss {avg_loss:.4f}")
    
        fold_model.eval()
        fold_likelihood.eval()
    
        test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
        means_list = []
        vars_list = []
    
        try:
            with torch.no_grad(), gpytorch.settings.num_likelihood_samples(NUM_SAMPLES):
                for x_batch, y_batch in test_loader:
                    x_batch = x_batch.to(device=device, dtype=torch.float32)
                    output = fold_model(x_batch)
                    means_list.append(output.mean.cpu())
                    vars_list.append(output.variance.cpu())
        except Exception as e:
            print(f"Skipping {j} - inference error: {e}")
            continue
    
        if len(means_list) == 0 or len(vars_list) == 0:
            print(f"Skipping {j} - no predictions")
            continue
    
        pred_means = torch.cat(means_list, dim=-1).mean(0)
        pred_vars = torch.cat(vars_list, dim=-1).mean(0)
    
        if torch.isnan(pred_means).any() or torch.isnan(pred_vars).any():
            print(f"Skipping {j} - NaN in predictions")
            continue
    
        positions = torch.sign(pred_means)
        strat_returns = positions * y_test_tensor
    
        if strat_returns.std() < 1e-8:
            print(f"Skipping {j} - zero variance")
            continue
    
        gross_sharpe = (strat_returns.mean() / strat_returns.std()) * (252 * 24 * 60) ** 0.5
        print(f"  {j} Gross Sharpe: {gross_sharpe.item():.3f}")
    
        fold_means.append(pred_means)
        fold_vars.append(pred_vars)
        fold_returns.append(y_test_tensor)
    
    if len(fold_means) == 0:
        print(f"No valid assets in fold {fold + 1}")
        continue
    
    eps = 1e-8
    T = min(m.shape[0] for m in fold_means)
    
    means_mat = torch.stack([m[:T] for m in fold_means], dim=1)
    vars_mat = torch.stack([v[:T] for v in fold_vars], dim=1)
    ret_mat = torch.stack([r[:T] for r in fold_returns], dim=1)
    
    scores = means_mat / (vars_mat + eps)

    # Long-only,  allocate to positive signal assets
    long_scores = torch.relu(scores)
    
    # Normalize to sum to 1
    # After computing weights, before portfolio returns
    avg_uncertainty = vars_mat.mean(dim=1)
    uncertainty_scalar = torch.exp(-avg_uncertainty)
    uncertainty_scalar = uncertainty_scalar / uncertainty_scalar.max()
    weights = weights * uncertainty_scalar.unsqueeze(1)
    weights = weights / (weights.sum(dim=1, keepdim=True) + eps)


    
    port_returns = (weights * ret_mat).sum(dim=1)
    turnover = torch.zeros(T, dtype=port_returns.dtype)
    turnover[1:] = weights[1:].sub(weights[:-1]).abs().sum(dim=1)
    
    port_returns_net = port_returns - 0.0005 * turnover
    
    fold_portfolio_returns.append(port_returns_net)
    fold_weights_list.append(weights)
    
    ann = 252 * 24 * 60
    fold_sharpe = (port_returns_net.mean() / port_returns_net.std()) * ann ** 0.5
    print(f"\nFold {fold + 1} Portfolio Sharpe: {fold_sharpe.item():.3f}")

print("\n" + "=" * 50)
print("WALK-FORWARD AGGREGATE RESULTS")

if len(fold_portfolio_returns) == 0:
    print("No valid folds")
else:
    all_returns = torch.cat(fold_portfolio_returns, dim=0)
    ann = 252 * 24 * 60
    eps = 1e-8
    
    ann_return = all_returns.mean() * ann
    ann_vol = all_returns.std() * ann ** 0.5
    sharpe = ann_return / (ann_vol + eps)
    
    downside = all_returns[all_returns < 0]
    if len(downside) > 1:
        sortino = ann_return / (downside.std() * ann ** 0.5 + eps)
    else:
        sortino = torch.tensor(float("nan"))
    
    cumulative = torch.cumprod(1 + all_returns, dim=0)
    running_max = torch.cummax(cumulative, dim=0).values
    drawdown = (cumulative - running_max) / (running_max + eps)
    max_drawdown = drawdown.min()
    calmar = ann_return / (max_drawdown.abs() + eps)
    
    print(f"Total periods evaluated: {len(all_returns)}")
    print(f"Valid folds:             {len(fold_portfolio_returns)}")
    print(f"\nAnnualised Return: {ann_return.item():.4f}")
    print(f"Annualised Vol:    {ann_vol.item():.4f}")
    print(f"Sharpe Ratio:      {sharpe.item():.3f}")
    print(f"Sortino Ratio:     {sortino.item():.3f}")
    print(f"Max Drawdown:      {max_drawdown.item():.3f}")
    print(f"Calmar Ratio:      {calmar.item():.3f}")
    
    print("\nPer-Fold Sharpes:")
    for i, r in enumerate(fold_portfolio_returns):
        fs = (r.mean() / r.std()) * (252 * 24 * 60) ** 0.5
        print(f"  Fold {i + 1}: {fs.item():.3f}")
    
    print("\n=== Baselines ===")
    all_fold_ret_mats = []
    
    for fold in range(N_FOLDS):
        train_start = fold * TEST_WINDOW
        train_end = train_start + TRAIN_WINDOW
        test_start = train_end
        test_end = test_start + TEST_WINDOW
    
        fold_ret_list = []
    
        for j in count:
            if j == "LUNA-USD" or j not in model_dict:
                continue
    
            try:
                full_data = train_dict[j]
                prices = full_data["price"].astype(float).values
                returns = np.diff(np.log(prices))
                y_test = returns[test_start:test_end]
    
                if len(y_test) > 0:
                    fold_ret_list.append(torch.tensor(y_test, dtype=torch.float32))
            except Exception:
                continue
    
        if len(fold_ret_list) > 0:
            T = min(r.shape[0] for r in fold_ret_list)
            ret_mat = torch.stack([r[:T] for r in fold_ret_list], dim=1)
            all_fold_ret_mats.append(ret_mat)


In [34]:
if len(all_fold_ret_mats) > 0:
    min_assets = min(r.shape[1] for r in all_fold_ret_mats)
    all_fold_ret_mats = [r[:,:min_assets] for r in all_fold_ret_mats]
    
    all_ret_mat = torch.cat(all_fold_ret_mats, dim=0)

    eq_returns = all_ret_mat.mean(dim=1)
    eq_sharpe = (eq_returns.mean() * ann) / (eq_returns.std() * ann ** 0.5 + eps)

    vol_scale = 1.0 / (all_ret_mat.std(dim=0) + eps)
    vol_weights = vol_scale / vol_scale.sum()
    vol_returns = (vol_weights * all_ret_mat).sum(dim=1)
    vol_sharpe = (vol_returns.mean() * ann) / (vol_returns.std() * ann ** 0.5 + eps)

    print(f"Equal Weight Sharpe: {eq_sharpe.item():.3f}")
    print(f"Vol-Scaled Sharpe:   {vol_sharpe.item():.3f}")
    print(f"\nDGP vs Equal Weight: {sharpe.item() - eq_sharpe.item():.3f}")
    print(f"DGP vs Vol-Scaled:   {sharpe.item() - vol_sharpe.item():.3f}")

Equal Weight Sharpe: 7.057
Vol-Scaled Sharpe:   6.969

DGP vs Equal Weight: 13.808
DGP vs Vol-Scaled:   13.897


In [ ]:
#Validation to see if there is an artifact in the training that is causing abnormal Sharpes
fold = 1
test_start = TRAIN_WINDOW + TEST_WINDOW
test_end = test_start + TEST_WINDOW
btc_prices = train_dict['BTC-USD']['price'].astype('float').values[test_start:test_end]
btc_returns = np.diff(np.log(btc_prices))
print(f"BTC vol in fold 2: {btc_returns.std():.6f}")
print(f"BTC max return: {np.max(np.abs(btc_returns)):.4f}")


BTC vol in fold 2: 0.006499
BTC max return: 0.0527


In [ ]:
#  Baseline Configuration 
TRAIN_WINDOW = 15000
TEST_WINDOW = 2500
N_FOLDS = 4
ann_factor = 252 * 24 * 60
eps = 1e-8
cost_per_trade = 0.0005

# Results storage 
xgb_fold_returns = []
lr_fold_returns = []

for fold in range(N_FOLDS):
    print(f"\n{'='*50}")
    print(f"BASELINE FOLD {fold+1}/{N_FOLDS}")
    
    train_start = fold * TEST_WINDOW
    train_end = train_start + TRAIN_WINDOW
    test_start = train_end
    test_end = test_start + TEST_WINDOW
    
    xgb_fold_means = []
    lr_fold_means = []
    fold_returns = []
    
    for j in count:
    
        if j == 'LUNA-USD':
            continue
    
        if j not in model_dict:
            continue
    
        try:
            full_data = train_dict[j]
            prices = full_data['price'].astype('float').values
            rates = full_data['rate'].astype('float').values
    
            returns = np.diff(np.log(prices))
            X = np.column_stack([rates[:-1], prices[:-1]])
    
            X_train = X[train_start:train_end]
            y_train = returns[train_start:train_end]
            X_test = X[test_start:test_end]
            y_test = returns[test_start:test_end]
    
            if len(X_train) < 1000 or len(X_test) < 100:
                continue
    
            # Scale
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
    
        except Exception as e:
            print(f"Skipping {j} - data error: {e}")
            continue
    
        # XGBoost 
        try:
            xgb = XGBRegressor(
                n_estimators=100,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                verbosity=0
            )
            xgb.fit(X_train_scaled, y_train)
            xgb_preds = xgb.predict(X_test_scaled)
            xgb_preds_tensor = torch.tensor(xgb_preds, dtype=torch.float32)
    
            # Per-asset XGBoost Sharpe
            xgb_positions = torch.sign(xgb_preds_tensor)
            xgb_strat = xgb_positions * torch.tensor(y_test, dtype=torch.float32)
            if xgb_strat.std() > 1e-8:
                xgb_sharpe = (xgb_strat.mean() / xgb_strat.std()) * ann_factor**0.5
                print(f"  {j} XGB Gross Sharpe: {xgb_sharpe.item():.3f}")
                xgb_fold_means.append(xgb_preds_tensor)
    
        except Exception as e:
            print(f"  {j} XGB error: {e}")
    
        #Ridge Regression 
        try:
            lr = Ridge(alpha=1.0)
            lr.fit(X_train_scaled, y_train)
            lr_preds = lr.predict(X_test_scaled)
            lr_preds_tensor = torch.tensor(lr_preds, dtype=torch.float32)
    
            # Per-asset LR Sharpe
            lr_positions = torch.sign(lr_preds_tensor)
            lr_strat = lr_positions * torch.tensor(y_test, dtype=torch.float32)
            if lr_strat.std() > 1e-8:
                lr_sharpe = (lr_strat.mean() / lr_strat.std()) * ann_factor**0.5
                print(f"  {j} LR  Gross Sharpe: {lr_sharpe.item():.3f}")
                lr_fold_means.append(lr_preds_tensor)
    
        except Exception as e:
            print(f"  {j} LR error: {e}")
    
        # Collect returns for portfolio
        y_test_tensor = torch.tensor(y_test, dtype=torch.float32)
        if y_test_tensor.std() > 1e-8:
            fold_returns.append(y_test_tensor)
    
    if len(fold_returns) == 0:
        print(f"No valid assets in fold {fold+1}")
        continue
    
    T = min(r.shape[0] for r in fold_returns)
    ret_mat = torch.stack([r[:T] for r in fold_returns], dim=1)
    
    # XGBoost Portfolio
    if len(xgb_fold_means) > 0:
        T_xgb = min(m.shape[0] for m in xgb_fold_means)
        T_xgb = min(T_xgb, T)
        xgb_means_mat = torch.stack([m[:T_xgb] for m in xgb_fold_means], dim=1)
        ret_mat_xgb = ret_mat[:T_xgb]
    
        # Long-only weights based on positive predictions
        xgb_scores = torch.relu(xgb_means_mat)
        xgb_weights = xgb_scores / (xgb_scores.sum(dim=1, keepdim=True) + eps)
        xgb_weights = torch.clamp(xgb_weights, min=0.0, max=0.20)
        xgb_weights = xgb_weights / (xgb_weights.sum(dim=1, keepdim=True) + eps)
    
        xgb_port = (xgb_weights * ret_mat_xgb).sum(dim=1)
        xgb_turnover = torch.zeros(T_xgb)
        xgb_turnover[1:] = xgb_weights[1:].sub(xgb_weights[:-1]).abs().sum(dim=1)
        xgb_port_net = xgb_port - cost_per_trade * xgb_turnover
    
        if xgb_port_net.std() > 1e-8:
            xgb_fold_sharpe = (xgb_port_net.mean() / xgb_port_net.std()) * ann_factor**0.5
            print(f"\nFold {fold+1} XGBoost Portfolio Sharpe: {xgb_fold_sharpe.item():.3f}")
            xgb_fold_returns.append(xgb_port_net)
    
    # Linear Regression Portfolio 
    if len(lr_fold_means) > 0:
        T_lr = min(m.shape[0] for m in lr_fold_means)
        T_lr = min(T_lr, T)
        lr_means_mat = torch.stack([m[:T_lr] for m in lr_fold_means], dim=1)
        ret_mat_lr = ret_mat[:T_lr]
    
        # Long-only weights based on positive predictions
        lr_scores = torch.relu(lr_means_mat)
        lr_weights = lr_scores / (lr_scores.sum(dim=1, keepdim=True) + eps)
        lr_weights = torch.clamp(lr_weights, min=0.0, max=0.20)
        lr_weights = lr_weights / (lr_weights.sum(dim=1, keepdim=True) + eps)
    
        lr_port = (lr_weights * ret_mat_lr).sum(dim=1)
        lr_turnover = torch.zeros(T_lr)
        lr_turnover[1:] = lr_weights[1:].sub(lr_weights[:-1]).abs().sum(dim=1)
        lr_port_net = lr_port - cost_per_trade * lr_turnover
    
        if lr_port_net.std() > 1e-8:
            lr_fold_sharpe = (lr_port_net.mean() / lr_port_net.std()) * ann_factor**0.5
            print(f"Fold {fold+1} Linear Reg Portfolio Sharpe: {lr_fold_sharpe.item():.3f}")
            lr_fold_returns.append(lr_port_net)

#  Aggregate Results 
print(f"\n{'='*50}")
print("BASELINE AGGREGATE RESULTS")

for name, fold_rets in [("XGBoost", xgb_fold_returns),
                         ("Linear Regression", lr_fold_returns)]:
    if len(fold_rets) == 0:
        print(f"{name}: No valid folds")
        continue
    
    all_rets = torch.cat(fold_rets, dim=0)
    ann_ret = all_rets.mean() * ann_factor
    ann_vol = all_rets.std() * ann_factor**0.5
    sharpe = ann_ret / (ann_vol + eps)
    
    downside = all_rets[all_rets < 0]
    sortino = ann_ret / (downside.std() * ann_factor**0.5 + eps)
    
    cumulative = torch.cumprod(1 + all_rets, dim=0)
    running_max = torch.cummax(cumulative, dim=0).values
    drawdown = (cumulative - running_max) / (running_max + eps)
    max_dd = drawdown.min()
    
    print(f"\n{name}:")
    print(f"  Annualised Return: {ann_ret.item():.4f}")
    print(f"  Annualised Vol:    {ann_vol.item():.4f}")
    print(f"  Sharpe Ratio:      {sharpe.item():.3f}")
    print(f"  Sortino Ratio:     {sortino.item():.3f}")
    print(f"  Max Drawdown:      {max_dd.item():.3f}")
    
    print(f"  Per-Fold Sharpes:")
    for i, r in enumerate(fold_rets):
        fs = (r.mean() / r.std()) * ann_factor**0.5
        print(f"    Fold {i+1}: {fs.item():.3f}")


BASELINE FOLD 1/4
  1INCH-USD XGB Gross Sharpe: 4.756
  1INCH-USD LR  Gross Sharpe: 25.011
  AAVE-USD XGB Gross Sharpe: 7.681
  AAVE-USD LR  Gross Sharpe: 63.462
  ADA-USD XGB Gross Sharpe: 6.840
  ADA-USD LR  Gross Sharpe: 12.041
  ALGO-USD XGB Gross Sharpe: 19.672
  ALGO-USD LR  Gross Sharpe: 20.297
  ATOM-USD XGB Gross Sharpe: 12.259
  ATOM-USD LR  Gross Sharpe: -3.658
  AVAX-USD XGB Gross Sharpe: 9.267
  AVAX-USD LR  Gross Sharpe: 3.572
  BCH-USD XGB Gross Sharpe: 18.656
  BCH-USD LR  Gross Sharpe: -2.481
  BTC-USD XGB Gross Sharpe: 2.611
  BTC-USD LR  Gross Sharpe: -10.155
  COMP-USD XGB Gross Sharpe: 8.654
  COMP-USD LR  Gross Sharpe: -6.715
  CRV-USD XGB Gross Sharpe: 21.447
  CRV-USD LR  Gross Sharpe: 8.293
  DOGE-USD XGB Gross Sharpe: 16.758
  DOGE-USD LR  Gross Sharpe: 26.882
  DOT-USD XGB Gross Sharpe: 9.634
  DOT-USD LR  Gross Sharpe: 10.110
  EOS-USD XGB Gross Sharpe: 49.422
  EOS-USD LR  Gross Sharpe: 12.207
  ETH-USD XGB Gross Sharpe: 6.400
  ETH-USD LR  Gross Sharpe: -

In [ ]:
# Run this after adding uncertainty scaling to get the full comparison
print("FINAL COMPARISON TABLE")
print(f"{'Method':<20} {'Sharpe':>8} {'Sortino':>8} {'MaxDD':>8} {'F1':>8} {'F2':>8} {'F3':>8}")
print("-" * 70)

methods = {
    'DGP (uncertainty)': fold_portfolio_returns,
    'XGBoost': xgb_fold_returns,
    'Linear Reg': lr_fold_returns
}

for name, fold_rets in methods.items():
    if len(fold_rets) == 0:
        continue
    all_rets = torch.cat(fold_rets, dim=0)
    ann = 252 * 24 * 60
    eps = 1e-8
    
    sharpe = (all_rets.mean() * ann) / (all_rets.std() * ann**0.5 + eps)
    downside = all_rets[all_rets < 0]
    sortino = (all_rets.mean() * ann) / (downside.std() * ann**0.5 + eps)
    cumulative = torch.cumprod(1 + all_rets, dim=0)
    running_max = torch.cummax(cumulative, dim=0).values
    max_dd = ((cumulative - running_max) / (running_max + eps)).min()
    
    fold_sharpes = []
    for r in fold_rets:
        fs = (r.mean() / r.std()) * ann**0.5
        fold_sharpes.append(fs.item())
    
    while len(fold_sharpes) < 3:
        fold_sharpes.append(float('nan'))
    
    print(f"{name:<20} {sharpe.item():>8.3f} {sortino.item():>8.3f} "
          f"{max_dd.item():>8.3f} {fold_sharpes[0]:>8.3f} "
          f"{fold_sharpes[1]:>8.3f} {fold_sharpes[2]:>8.3f}")


=== FINAL COMPARISON TABLE ===
Method                 Sharpe  Sortino    MaxDD       F1       F2       F3
----------------------------------------------------------------------
DGP (uncertainty)      20.865   30.574   -0.248   27.097  153.770  -12.513
XGBoost                35.376   50.265   -0.204   59.477  162.447   -5.402
Linear Reg             32.115   43.959   -0.194   51.360  149.110   -4.842
